In [ ]:
!pip install crewai crewai_tools

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY']=userdata.get('GOOGLE_API_KEY')

# Basic Flow (Vienkārša plūsma)

In [ ]:
!pip install nest_asyncio

In [ ]:
! pip install litellm

In [ ]:
import nest_asyncio
from crewai.flow.flow import Flow, listen, start
from dotenv import load_dotenv
from litellm import completion
import os

# Allow nested event loops
nest_asyncio.apply()

class ExampleFlow(Flow):
    model = "gemini/gemini-2.0-flash"


    @start()
    def generate_city(self):
        print("Starting flow")

        response = completion(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": "Return the name of a random city in the world.",
                },
            ],
            api_key=os.getenv('GEMINI_API_KEY') # Explicitly pass the API key
        )

        random_city = response["choices"][0]["message"]["content"]
        print(f"Random City: {random_city}")

        return random_city

    @listen(generate_city)
    def generate_fun_fact(self, random_city):
        response = completion(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": f"Tell me a fun fact about {random_city}",
                },
            ],
            api_key=os.getenv('GEMINI_API_KEY') # Explicitly pass the API key
        )

        fun_fact = response["choices"][0]["message"]["content"]
        return fun_fact


flow = ExampleFlow()
result = flow.kickoff()

print(f"Generated fun fact: {result}")

# Unstructured Flow (Nestrukturizēta plūsma)

In [ ]:
from crewai.flow.flow import Flow, listen, start
import nest_asyncio

nest_asyncio.apply()

class UntructuredExampleFlow(Flow):

    @start()
    def first_method(self):
        print("Starting flow")
        # Avoid printing the entire self.state object directly to prevent recursion
        # print(f"State before first_method:\n{self.state}")
        print("State before first_method: Initializing state...")
        self.state["message"] = "Hello from unstructured flow"
        self.state["counter"] = 0

    @listen(first_method)
    def second_method(self):
        # Avoid printing the entire self.state object directly
        # print(f"State before second_method:\n{self.state}")
        print(f"State before second_method: message={self.state.get('message')}, counter={self.state.get('counter')}")
        self.state["message"] += " - updated"
        self.state["counter"] += 1

    @listen(second_method)
    def third_method(self):
        # Avoid printing the entire self.state object directly
        # print(f"State before third_method:\n{self.state}")
        print(f"State before third_method: message={self.state.get('message')}, counter={self.state.get('counter')}")
        self.state["message"] += " - updated again"
        self.state["counter"] += 1

        # You can print specific parts of the state or convert the dictionary to a string
        print(f"State after third_method: {str(self.state)}")


flow = UntructuredExampleFlow()
flow.kickoff()

# Structured Flow (Strukturizēta plūsma)

In [ ]:
from crewai.flow.flow import Flow, listen, start
from pydantic import BaseModel
import nest_asyncio

nest_asyncio.apply()

class ExampleState(BaseModel):
    counter: int = 0
    message: str = ""


class StructuredExampleFlow(Flow[ExampleState]):

    @start()
    def first_method(self):
        print("Starting flow")
        print(f"State before first_method:\n{self.state}\n")
        self.state.message = "Hello from structured flow"
        self.state.counter += 1

    @listen(first_method)
    def second_method(self):
        print(f"State before second_method:\n{self.state}\n")
        self.state.counter += 1
        self.state.message += " - updated"

    @listen(second_method)
    def third_method(self):
        print(f"State before third_method:\n{self.state}\n")
        self.state.counter += 1
        self.state.message += " - updated again"

        print(f"State after third_method: {self.state}")


flow = StructuredExampleFlow()
flow.kickoff()

print(f"Final state:\n{flow.state}")

# Conditional Flows (Nosacījumu plūsmas)

## OR Flow (VAI plūsma)

In [ ]:
from crewai.flow.flow import Flow, listen, or_, start

import nest_asyncio

nest_asyncio.apply()

class OrExampleFlow(Flow):

    @start()
    def start_method(self):
        print("Starting flow")
        return "Hello from the start method"

    @listen(start_method)
    def second_method(self):
        print("Second method")
        return "Hello from the second method"

    @listen(or_(start_method, second_method))
    def logger(self, result):
        print(f"Logger: {result}")


flow = OrExampleFlow()
flow.kickoff()

## AND Flow (UN plūsma)

In [ ]:
from crewai.flow.flow import Flow, and_, listen, start

import nest_asyncio

nest_asyncio.apply()

class AndExampleFlow(Flow):

    @start()
    def start_method(self):
        print("---- Start Method ----")
        self.state["greeting"] = "Hello from the start method"

    @listen(start_method)
    def second_method(self):
        print("---- Second Method ----")
        self.state["joke"] = "What do computers eat? Microchips."

    @listen(and_(start_method, second_method))
    def logger(self):
        print("---- Logger ----")
        print(self.state)


flow = AndExampleFlow()
flow.kickoff()

## Router Flow (Maršrutēšanas plūsma)

In [ ]:
import random

from crewai.flow.flow import Flow, listen, router, start
from pydantic import BaseModel

import nest_asyncio

nest_asyncio.apply()

class ExampleState(BaseModel):
    success_flag: bool = False


class RouterFlow(Flow[ExampleState]):

    @start()
    def start_method(self):
        print("Starting the structured flow")
        random_boolean = random.choice([True, False])
        self.state.success_flag = random_boolean

    @router(start_method)
    def second_method(self):
        if self.state.success_flag:
            return "success"
        else:
            return "failed"

    @listen("success")
    def third_method(self):
        print("Third method running")

    @listen("failed")
    def fourth_method(self):
        print("Fourth method running")


flow = RouterFlow()
flow.kickoff()

In [ ]:
flow.plot()

# Flaws with crews (Plūsmas ar komandām)

In [ ]:
!pip install patool

In [ ]:
import patoolib
patoolib.extract_archive("/content/config.zip")

In [ ]:
# Define file paths for YAML configurations
import os
import json
import yaml

files = {
    'agents': '/content/config/agents.yaml',
    'tasks': '/content/config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY']=userdata.get('GOOGLE_API_KEY')



In [ ]:
from crewai import Agent, Crew, Process, Task
poem_writer = Agent(
    config=agents_config["poem_writer"],
    llm = "gemini/gemini-2.0-flash",
)

In [ ]:
write_poem = Task(
    config=tasks_config["write_poem"],
    agent=poem_writer,
)

In [ ]:
poem_crew = Crew(
    agents=[poem_writer],
    tasks=[write_poem],
    process=Process.sequential,
    verbose=False,
)


In [ ]:
from random import randint

from crewai.flow.flow import Flow, listen, start
from pydantic import BaseModel

import nest_asyncio

# Allow nested event loops
nest_asyncio.apply()

class PoemState(BaseModel):
    sentence_count: int = 1
    poem: str = ""


class PoemFlow(Flow[PoemState]):

    @start()
    def generate_sentence_count(self):
        print("Generating sentence count")
        self.state.sentence_count = randint(1, 5)

    @listen(generate_sentence_count)
    def generate_poem(self):
        print("Generating poem")
        result = (
            poem_crew
            .kickoff(
                inputs={
                    "sentence_count": self.state.sentence_count,
                }
            )
        )

        print("Poem generated", result.raw)
        self.state.poem = result.raw

    @listen(generate_poem)
    def save_poem(self):
        print("Saving poem")
        with open("poem.txt", "w") as f:
            f.write(self.state.poem)


def kickoff():
    poem_flow = PoemFlow()
    poem_flow.kickoff()

if __name__ == "__main__":
    kickoff()

In [ ]:
poem_flow = PoemFlow()
poem_flow.plot("my_flow_plot")